# 04 - Baseline Model

**Day 1 - Cross-Temporal Hybrid NIDS**

Trains and evaluates the Day 1 lightweight Random Forest baseline on the
chronologically-split train/test parquet files produced by
`scripts/prepare_data.py`. Hyperparameters mirror
`scripts/train_baseline.py` exactly, so results here are reproducible
via the command-line script as well.

No hyperparameter tuning, no cross-validation grid search, no SHAP -
those are out of Day 1 scope. This is a baseline only.

> **Status:** this notebook has **not** been executed against real data.
> Run `scripts/prepare_data.py` first to produce
> `data/processed/day1/{train,test}.parquet`, then run this notebook.

In [ ]:
# --- Setup -----------------------------------------------------------
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "day1"
MODELS_DIR = PROJECT_ROOT / "models" / "day1"
RESULTS_DIR = PROJECT_ROOT / "results" / "day1"

TRAIN_PATH = PROCESSED_DIR / "train.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"
METADATA_PATH = PROCESSED_DIR / "split_metadata.json"

# Lightweight settings for an 8GB RAM, CPU-only machine - mirrors
# scripts/train_baseline.py defaults exactly.
N_ESTIMATORS = 100
MAX_DEPTH = 20
N_JOBS = -1
RANDOM_STATE = 42

## 1. Loading the prepared data

Loads the already-split, already-cleaned train/test Parquet files (small
relative to the raw dataset) plus the split metadata (feature names,
timestamp column, leakage-check result).

In [ ]:
for path in (TRAIN_PATH, TEST_PATH, METADATA_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run scripts/prepare_data.py first to "
            "produce the chronological train/test split."
        )

metadata = json.loads(METADATA_PATH.read_text())
feature_names = metadata["feature_names"]

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape : {test_df.shape}")
print(f"Feature count: {len(feature_names)}")
print(f"Timestamp column used for split: {metadata.get('timestamp_column')!r}")
print(f"Leakage check passed: {metadata.get('leakage_check', {}).get('passed')}")

In [ ]:
X_train = train_df[feature_names]
y_train = train_df["label_binary"]
X_test = test_df[feature_names]
y_test = test_df["label_binary"]

# NaNs are intentionally left as NaN through cleaning (see src/data/cleaner.py);
# RandomForestClassifier cannot handle NaN, so impute here at the
# model boundary with train-set medians (same strategy as
# scripts/train_baseline.py).
n_nan_train = int(X_train.isna().to_numpy().sum())
n_nan_test = int(X_test.isna().to_numpy().sum())
print(f"NaNs before imputation - train: {n_nan_train}, test: {n_nan_test}")

if n_nan_train or n_nan_test:
    medians = X_train.median(numeric_only=True)
    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)
    print("Imputed NaNs with train-set column medians.")

## 2. Training the Day 1 Random Forest baseline

Deliberately modest settings (`n_estimators=100`, `max_depth=20`) so
training stays fast and memory-bounded on an 8 GB RAM, CPU-only machine.
`class_weight="balanced"` compensates for the typical benign/attack class
imbalance in CIC-IDS2017 without any resampling.

In [ ]:
clf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)

start = time.time()
clf.fit(X_train, y_train)
train_seconds = time.time() - start

print(f"Training complete in {train_seconds:.1f}s")

## 3. Predictions

In [ ]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

pd.Series(y_pred).value_counts().rename({0: "predicted benign (0)", 1: "predicted attack (1)"})

## 4. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}")
print(f"FN={fn}  TP={tp}")

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["benign (0)", "attack (1)"]).plot(ax=ax, colorbar=False)
ax.set_title("Confusion matrix - Random Forest baseline")
plt.tight_layout()
plt.show()

## 5. Precision, Recall, F1

Positive class = attack (`label_binary == 1`).

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")

## 6. ROC-AUC and PR-AUC

In [ ]:
try:
    roc_auc = roc_auc_score(y_test, y_proba)
except ValueError as exc:
    roc_auc = float("nan")
    print(f"ROC-AUC could not be computed: {exc}")

try:
    pr_auc = average_precision_score(y_test, y_proba)
except ValueError as exc:
    pr_auc = float("nan")
    print(f"PR-AUC could not be computed: {exc}")

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC : {pr_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

fpr_curve, tpr_curve, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr_curve, tpr_curve, label=f"ROC-AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve")
axes[0].legend()

prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec_curve, prec_curve, label=f"PR-AUC = {pr_auc:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. False Positive Rate / False Negative Rate

In [ ]:
fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
fnr = fn / (fn + tp) if (fn + tp) > 0 else float("nan")

print(f"False Positive Rate: {fpr:.4f}")
print(f"False Negative Rate: {fnr:.4f}")

## 8. Feature importance

Gini-based feature importances from the fitted `RandomForestClassifier`
(the same model used above) - not a substitute for SHAP/explainability
work, which is explicitly out of Day 1 scope.

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
top_n = importances.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
top_n.iloc[::-1].plot(kind="barh", ax=ax)
ax.set_title("Top 20 feature importances - Random Forest baseline")
ax.set_xlabel("Gini importance")
plt.tight_layout()
plt.show()

top_n

## 9. Saving the baseline model

Mirrors the outputs of `scripts/train_baseline.py` exactly:
`models/day1/random_forest_baseline.joblib` plus metadata, and
`results/day1/baseline_metrics.json`.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics = {
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "false_positive_rate": float(fpr),
    "false_negative_rate": float(fnr),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}

model_path = MODELS_DIR / "random_forest_baseline.joblib"
joblib.dump(clf, model_path)
print(f"Saved model -> {model_path}")

model_metadata = {
    "model_type": "RandomForestClassifier",
    "hyperparameters": {
        "n_estimators": N_ESTIMATORS,
        "max_depth": MAX_DEPTH,
        "random_state": RANDOM_STATE,
        "class_weight": "balanced",
    },
    "feature_names": feature_names,
    "label_col": "label_binary",
    "timestamp_column": metadata.get("timestamp_column"),
    "split_summary": metadata.get("split_summary"),
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "train_seconds": train_seconds,
    "nan_imputation": {
        "strategy": "train-median",
        "train_nans_before_impute": n_nan_train,
        "test_nans_before_impute": n_nan_test,
    },
}
(MODELS_DIR / "random_forest_baseline_metadata.json").write_text(
    json.dumps(model_metadata, indent=2, default=str)
)

results_path = RESULTS_DIR / "baseline_metrics.json"
results_path.write_text(json.dumps(metrics, indent=2, default=str))
print(f"Saved evaluation metrics -> {results_path}")

metrics

## Summary

* Loaded the chronologically-split train/test data + metadata.
* Trained the same lightweight Random Forest baseline as
  `scripts/train_baseline.py` (`n_estimators=100`, `max_depth=20`,
  `class_weight="balanced"`).
* Reported confusion matrix, Precision, Recall, F1, ROC-AUC, PR-AUC,
  FPR, FNR, and feature importance.
* Saved the model and metrics to the same locations the command-line
  script would use.

This concludes the Day 1 notebook set: dataset inspection ->
preprocessing -> temporal split -> baseline model.